# Notebook 04 — Generate Embeddings

This notebook encodes all artifacts (papers, patents, projects, parts) as
dense vector embeddings using a sentence-transformer model.

**What it does:**
1. Loads all processed CSVs from `data/processed/`
2. Combines them into a single DataFrame
3. Generates embeddings for each artifact's `text` field
4. Caches results in `data/embeddings/embeddings.json` (safe to re-run)

**Before running:** complete notebooks 01-03 first to generate the CSVs.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from pathlib import Path
from src.utils.config import load_config
from src.embed.embeddings import load_model, generate_embeddings

cfg = load_config()

In [ ]:
# --- 1. Load all processed CSVs ---

processed_dir = Path('../data/processed')
dfs = []
for csv_file in ['papers.csv', 'patents.csv', 'projects.csv', 'parts.csv']:
    path = processed_dir / csv_file
    if path.exists():
        df = pd.read_csv(path)
        dfs.append(df)
        print(f'Loaded {len(df)} records from {csv_file}')
    else:
        print(f'Not found: {csv_file} (skipping)')

combined = pd.concat(dfs, ignore_index=True)
print(f'\nTotal: {len(combined)} artifacts across {combined["type"].nunique()} types')

In [ ]:
# --- 2. Load embedding model ---
# The model is downloaded on first use (~80 MB for all-MiniLM-L6-v2)

model = load_model(cfg['embedding']['model'])
print('Model loaded.')

In [ ]:
# --- 3. Generate and cache embeddings ---
# Already-embedded items are loaded from cache and skipped.

cache_file = Path('../data/embeddings/embeddings.json')

embedding_cache = generate_embeddings(
    df=combined,
    model=model,
    text_col=cfg['embedding']['text_field'],
    id_col='id',
    cache_file=cache_file,
    batch_size=cfg['embedding']['batch_size'],
)

print(f'Embedding cache has {len(embedding_cache)} entries.')